In [18]:
import numpy as np
import pandas as pd
from mhcgnomes import parse

# 1 Read total data

In [19]:
#DATA_PATH = '/home/denklewer/Data/' #ubuntu
EXPERIMENT_NAME = 'length_dependent_hmm'
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/' # windows

IEDB_DATA_PATH = DATA_PATH + 'Binding data/IEDB/mhc_ligand_full_single_file/mhc_ligand_full.csv'
NetMHCPan_41_DATA_PATH = DATA_PATH + 'Binding data/NetMHCpan-4.1/NetMHCpan_train/'


In [20]:
iedb_df = pd.read_csv(IEDB_DATA_PATH, sep=',', header=[0,1])


D:\Tools\Python\Anaconda\envs\pomegranate\lib\site-packages\IPython\core\interactiveshell.py:3444: DtypeWarning: Columns (5,14,23,24,25,26,29,30,31,32,33,34,35,36,37,38,41,42,43,44,45,46,47,48,49,50,53,54,55,56,57,58,59,60,61,62,63,64,65,68,69,70,71,72,73,74,75,76,77,78,81,84,89,90,91,92,93,94,97,99) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [21]:
len(iedb_df)

3235984

In [33]:
ALLELES = ['HLA-A*02:01', 'HLA-B*27:05', 'HLA-B*07:02', 'HLA-A*03:01', 'HLA-A*01:01', 'HLA-DRB1*01:01', 'HLA-DRB1*04:01', 'HLA-DPA1*01:03/DPB1*04:01']

In [26]:
def tuple_to_last_elem(column):
    return column[-1]

def filter_iedb_dataset(iedb_df, include_qualitative=True, include_mass_spec=True):
    QUALITATIVE_TO_AFFINITY_AND_INEQUALITY = {
    "Negative": (5000.0, ">"),
    "Positive": (500.0, "<"),  # used for mass-spec hits
    "Positive-High": (100.0, "<"),
    "Positive-Intermediate": (1000.0, "<"),
    "Positive-Low": (5000.0, "<"),
    }
    QUALITATIVE_TO_AFFINITY = dict(
        (key, value[0]) for (key, value)
        in QUALITATIVE_TO_AFFINITY_AND_INEQUALITY.items())
    QUALITATIVE_TO_INEQUALITY = dict(
        (key, value[1]) for (key, value)
        in QUALITATIVE_TO_AFFINITY_AND_INEQUALITY.items())
    
    
    
    print("Loaded iedb data: %s" % str(iedb_df.shape))
    filtered_df = iedb_df.copy()
    filtered_df = filtered_df[filtered_df[('Epitope', 'Object Type')] == "Linear peptide"]
    print("Linear peptides only, remaining: %d" % len(filtered_df))
    
    ## Clear bad data
    print("Subselecting to valid peptides (dropping peptides with PTMs). Starting with: %d" % len(filtered_df))
    filtered_df[('Epitope', 'Description')] = filtered_df[('Epitope', 'Description')].str.strip()
    filtered_df = filtered_df.loc[
        iedb_df[('Epitope', 'Description')].str.match("^[ACDEFGHIKLMNPQRSTVWY]+$")
    ]
    print("Now: %d" % len(filtered_df))
    
    
#     filtered_df = filtered_df[~filtered_df[('Epitope', 'Description')].str.contains("\+")]
#     filtered_df = filtered_df[~filtered_df[('Epitope', 'Description')].str.contains("X")]
#     filtered_df = filtered_df[~filtered_df[('Epitope', 'Description')].str.contains("B")]
#     filtered_df = filtered_df[~filtered_df[('Epitope', 'Description')].str.contains("Z")]
    filtered_df = filtered_df[[
    ('Epitope', 'Object Type'),
    ('Epitope', 'Description'),
    ('Epitope', 'Parent Species'),
    ('Assay', 'Units'),
    ('Assay', 'Qualitative Measure'),
    ('Assay', 'Quantitative measurement'),
     ('Assay', 'Measurement Inequality'),
    ('MHC', 'Allele Name'),
    ('MHC', 'Allele IRI'),
    ('MHC', 'MHC allele class'),
    ('Assay', 'Assay Group'),
    ('Assay', 'Method/Technique')    
             ]]
    print("Subselecting assay groups as in Sarkizova. Starting with: %d" % len(filtered_df))

    assay_type1 = (filtered_df[('Assay', 'Assay Group')] == 'half maximal effective concentration (EC50)') & \
                                              (filtered_df[('Assay', 'Method/Technique')] == 'purified MHC/direct/fluorescence')
    assay_type2 = (filtered_df[('Assay', 'Assay Group')] =='dissociation constant KD') & \
                                                  (filtered_df[('Assay', 'Method/Technique')] == 'purified MHC/direct/radioactivity')
    assay_type3 = (filtered_df[('Assay', 'Assay Group')] == 'half maximal effective concentration (EC50)') & \
                                                  (filtered_df[('Assay', 'Method/Technique')] == 'cellular MHC/direct/fluorescence')
    filtered_df = filtered_df[~assay_type1]
    filtered_df = filtered_df[~assay_type2]
    filtered_df = filtered_df[~assay_type3]
    filtered_df.columns = filtered_df.columns.map(tuple_to_last_elem)
    print("Now: %d" % len(filtered_df))
    
    
    quantitative = filtered_df.loc[filtered_df["Units"] == "nM"].copy()
    quantitative["measurement_type"] = "quantitative"
    quantitative = quantitative.rename(columns={"Measurement Inequality":"measurement_inequality"})
    quantitative['measurement_inequality'] = quantitative['measurement_inequality'].fillna("=").map(lambda s: {">=": ">", "<=": "<"}.get(s, s))
    print("Quantitative measurements: %d" % len(quantitative))
    # Ilya: remove quantitative measurement with no measurement values
    print("Dropping quantitative measurements with no measurement value")
    quantitative.dropna(subset = ["Quantitative measurement"], inplace=True)
    print("Quantitative measurements: %d" % len(quantitative))
    
    qualitative = filtered_df.loc[filtered_df["Units"] != "nM"].copy()
    qualitative["measurement_type"] = "qualitative"
    print("Qualitative measurements: %d" % len(qualitative))
    if not include_mass_spec:
        qualitative = qualitative.loc[
            (~qualitative["Method/Technique"].str.contains("mass spec"))
        ].copy()

    qualitative["Quantitative measurement"] = (
        qualitative["Qualitative Measure"].map(QUALITATIVE_TO_AFFINITY))
    qualitative["measurement_inequality"] = (qualitative["Qualitative Measure"].map(QUALITATIVE_TO_INEQUALITY))

    print("Qualitative measurements (possibly after dropping MS): %d" % (
        len(qualitative)))
    
    filtered_df = pd.concat(
        (
            ([quantitative]) +
            ([qualitative] if include_qualitative else [])),
        ignore_index=True)
    print("Now: %d" % len(filtered_df))
    
    ## Mark binders
    binder_list = ['Positive', 'Positive-High', 'Positive-Intermediate', 'Positive-Low']
    non_binder_list = ['Negative']
    filtered_df.loc[filtered_df['Quantitative measurement'] <= 500, 'binder'] = 1
    filtered_df.loc[filtered_df['Quantitative measurement'] > 500, 'binder'] = 0

    filtered_df.loc[filtered_df['Qualitative Measure'].isin(binder_list),'binder'] = 1
    filtered_df.loc[filtered_df['Qualitative Measure'].isin(non_binder_list),'binder'] = 0
    
    filtered_df.binder = filtered_df.binder.astype(int)
    filtered_df = filtered_df.drop(columns=[
     'Qualitative Measure',
     'Units',
     'Object Type',
     'Allele IRI'
     ])
    
    filtered_df = filtered_df.rename(columns={"Allele Name":"allele", 
                                              "Description":"peptide",
                                              "Quantitative measurement":"measurement_value"})
    filtered_df['length'] = filtered_df['peptide'].str.len()
    
    #filtered_df['allele'] = filtered_df['original_allele'].map(parse).to_string()
    
    print("Take unique (peptide, allele, length, 'measurement_value', 'measurement_inequality', 'measurement_type', binder). Starting with: %d" % len(filtered_df))

    filtered_df = filtered_df.drop_duplicates([
    'peptide',
    'allele',
    'length',
    'measurement_value',
    'measurement_inequality',
    'measurement_type',
    'binder']).reset_index(drop=True)
    print("Now: %d" % len(filtered_df))

    
    print("Allele_distribution")
    print(filtered_df['allele'].value_counts()[:30])
    


    assert 'peptide' in filtered_df.columns
    assert 'allele' in filtered_df.columns
    assert 'length' in filtered_df.columns
    assert 'binder' in filtered_df.columns
    return filtered_df

In [27]:
iedb_df = filter_iedb_dataset(iedb_df)


Loaded iedb data: (3235984, 100)
Linear peptides only, remaining: 3235229
Subselecting to valid peptides (dropping peptides with PTMs). Starting with: 3235229
Now: 3104634
Subselecting assay groups as in Sarkizova. Starting with: 3104634


C:\Users\Denis\AppData\Local\Temp/ipykernel_18984/2719161792.py:62: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_df = filtered_df[~assay_type2]
C:\Users\Denis\AppData\Local\Temp/ipykernel_18984/2719161792.py:63: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_df = filtered_df[~assay_type3]


Now: 3103070
Quantitative measurements: 235469
Dropping quantitative measurements with no measurement value
Quantitative measurements: 235321
Qualitative measurements: 2867601
Qualitative measurements (possibly after dropping MS): 2867601
Now: 3102922
Take unique (peptide, allele, length, 'measurement_value', 'measurement_inequality', 'measurement_type', binder). Starting with: 3102922
Now: 1545780
Allele_distribution
HLA class I                  267328
HLA class II                 138388
HLA-B*27:05                   80229
HLA-A*02:01                   62231
HLA-B*07:02                   28948
HLA-DR                        28569
HLA-B*15:02                   21929
HLA-B*57:01                   21454
HLA-A*01:01                   21075
HLA-DRB1*01:01                19757
HLA-A*24:02                   18757
HLA-B*15:01                   18363
HLA-B*51:01                   17089
HLA-A*11:01                   16081
HLA-B*40:02                   15939
HLA-A*03:01                   15081
HL

In [29]:
target_df = iedb_df[iedb_df.allele == 'HLA-DPA1*01:03/DPB1*04:01']

In [32]:
target_df.sample(3000)['peptide'].to_csv("gibbs_input.csv", index=False, header=False)

In [16]:
iedb_df.allele.value_counts()

HLA class I                             267328
HLA class II                            138388
HLA-B*27:05                              80229
HLA-A*02:01                              62231
HLA-B*07:02                              28948
                                         ...  
HLA-DRB1*13:05                               1
HLA-B*08:01 N80I mutant                      1
BoLA-6*013:02                                1
HLA-B*57:01 I80N mutant                      1
HLA-B*35:08 Q65A, T69A, Q155A mutant         1
Name: allele, Length: 720, dtype: int64

In [17]:
iedb_df[iedb_df.allele == 'HLA-DPA1*01:03/DPB1*04:01']

,peptide,Parent Species,measurement_value,measurement_inequality,allele,MHC allele class,Assay Group,Method/Technique,measurement_type,Measurement Inequality,binder,length
118547,VLQAGFFLITRILTIPQSLD,Hepatitis B virus,10000.0,=,HLA-DPA1*01:03/DPB1*04:01,II,half maximal inhibitory concentration (IC50),purified MHC/competitive/fluorescence,quantitative,NaN,1,20
118548,MEVGWYRSPFSRVVHLYRNGK,Mus musculus,1000.0,=,HLA-DPA1*01:03/DPB1*04:01,II,half maximal inhibitory concentration (IC50),purified MHC/competitive/fluorescence,quantitative,NaN,1,21
118549,MEVGWYRPPFSRFVHLYRNGK,NaN,5000.0,=,HLA-DPA1*01:03/DPB1*04:01,II,half maximal inhibitory concentration (IC50),purified MHC/competitive/fluorescence,quantitative,NaN,1,21
118550,HSLGKWLGHPDKF,Homo sapiens,832000.0,=,HLA-DPA1*01:03/DPB1*04:01,II,half maximal inhibitory concentration (IC50),purified MHC/competitive/fluorescence,quantitative,NaN,1,13
118551,GGFMTTAFQYIIDNKG,Homo sapiens,2000000.0,>,HLA-DPA1*01:03/DPB1*04:01,II,half maximal inhibitory concentration (IC50),purified MHC/competitive/fluorescence,quantitative,NaN,0,16
...,...,...,...,...,...,...,...,...,...,...,...,...
1402963,RPRGFAYVQFEDVRDAEDALHN,Homo sapiens,500.0,<,HLA-DPA1*01:03/DPB1*04:01,II,ligand presentation,cellular MHC/mass spectrometry,qualitative,NaN,1,22
1403357,SDEMNVKVLDFEHFLPMLQ,Homo sapiens,500.0,<,HLA-DPA1*01:03/DPB1*04:01,II,ligand presentation,cellular MHC/mass spectrometry,qualitative,NaN,1,19
1403445,LFRKFHYLPFLPS,Homo sapiens,500.0,<,HLA-DPA1*01:03/DPB1*04:01,II,ligand presentation,cellular MHC/mass spectrometry,qualitative,NaN,1,13
1403559,SPEDEEKNFDQTRFLEDSLLNW,Homo sapiens,500.0,<,HLA-DPA1*01:03/DPB1*04:01,II,ligand presentation,cellular MHC/mass spectrometry,qualitative,NaN,1,22


# Make per allele DF

In [34]:
def make_per_allele_df(source_df, ALLELES):
    per_allele_df = dict()
    for allele_name in ALLELES:
        allele_df = source_df[source_df['allele'] == allele_name] 
        print(f"------Allele {allele_name} ----------")
        print("Binders_distribution:")
        print(allele_df['binder'].value_counts())
        print("Length_distribution:")
        print(allele_df['length'].value_counts()[:10])
        per_allele_df[allele_name] = allele_df
    return per_allele_df
        
        
per_allele_df = make_per_allele_df(iedb_df, ALLELES)    

------Allele HLA-A*02:01 ----------
Binders_distribution:
1    52768
0     9463
Name: binder, dtype: int64
Length_distribution:
9     37495
10    10713
11     3944
8      3073
12     1367
7      1269
13      913
15      784
14      710
16      533
Name: length, dtype: int64
------Allele HLA-B*27:05 ----------
Binders_distribution:
1    57428
0    22801
Name: binder, dtype: int64
Length_distribution:
9     31427
10    15052
11    10013
12     5348
13     3542
14     2678
15     2242
16     1867
17     1643
18     1271
Name: length, dtype: int64
------Allele HLA-B*07:02 ----------
Binders_distribution:
1    25566
0     3382
Name: binder, dtype: int64
Length_distribution:
9     16321
10     4861
11     2327
8      2092
12     1004
13      663
14      477
15      423
16      235
17      172
Name: length, dtype: int64
------Allele HLA-A*03:01 ----------
Binders_distribution:
1    10604
0     4477
Name: binder, dtype: int64
Length_distribution:
9     10306
10     3245
11      889
12      259

#  Save per allele DF

In [35]:
import mhcnames

TARGET_PATH_TO_FILES = DATA_PATH + EXPERIMENT_NAME + '/' 

for allele_name in ALLELES:
    total_df = per_allele_df[allele_name]
    compact_name = mhcnames.compact_allele_name(allele_name)
    # allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}data_{}.csv".format(TARGET_PATH_TO_FILES,compact_name),sep=";", index=False)

In [36]:
total_df

,peptide,Parent Species,measurement_value,measurement_inequality,allele,MHC allele class,Assay Group,Method/Technique,measurement_type,Measurement Inequality,binder,length
118547,VLQAGFFLITRILTIPQSLD,Hepatitis B virus,10000.0,=,HLA-DPA1*01:03/DPB1*04:01,II,half maximal inhibitory concentration (IC50),purified MHC/competitive/fluorescence,quantitative,NaN,1,20
118548,MEVGWYRSPFSRVVHLYRNGK,Mus musculus,1000.0,=,HLA-DPA1*01:03/DPB1*04:01,II,half maximal inhibitory concentration (IC50),purified MHC/competitive/fluorescence,quantitative,NaN,1,21
118549,MEVGWYRPPFSRFVHLYRNGK,NaN,5000.0,=,HLA-DPA1*01:03/DPB1*04:01,II,half maximal inhibitory concentration (IC50),purified MHC/competitive/fluorescence,quantitative,NaN,1,21
118550,HSLGKWLGHPDKF,Homo sapiens,832000.0,=,HLA-DPA1*01:03/DPB1*04:01,II,half maximal inhibitory concentration (IC50),purified MHC/competitive/fluorescence,quantitative,NaN,1,13
118551,GGFMTTAFQYIIDNKG,Homo sapiens,2000000.0,>,HLA-DPA1*01:03/DPB1*04:01,II,half maximal inhibitory concentration (IC50),purified MHC/competitive/fluorescence,quantitative,NaN,0,16
...,...,...,...,...,...,...,...,...,...,...,...,...
1402963,RPRGFAYVQFEDVRDAEDALHN,Homo sapiens,500.0,<,HLA-DPA1*01:03/DPB1*04:01,II,ligand presentation,cellular MHC/mass spectrometry,qualitative,NaN,1,22
1403357,SDEMNVKVLDFEHFLPMLQ,Homo sapiens,500.0,<,HLA-DPA1*01:03/DPB1*04:01,II,ligand presentation,cellular MHC/mass spectrometry,qualitative,NaN,1,19
1403445,LFRKFHYLPFLPS,Homo sapiens,500.0,<,HLA-DPA1*01:03/DPB1*04:01,II,ligand presentation,cellular MHC/mass spectrometry,qualitative,NaN,1,13
1403559,SPEDEEKNFDQTRFLEDSLLNW,Homo sapiens,500.0,<,HLA-DPA1*01:03/DPB1*04:01,II,ligand presentation,cellular MHC/mass spectrometry,qualitative,NaN,1,22


In [37]:
total_df['binder'].value_counts()

1    14225
0      462
Name: binder, dtype: int64